In [2]:
#Business Challenge 4: Agentic Model Risk Factory
# # 1.	First, we load our raw dataset and clean the data. The dataset contains undocumented codes 
# for education and marriage – we map them into clear category called “Other” to prevent model errors.
# # 2.	Split the data into training and testing. Training to 80%, testing to 20%
# # 3.	random_state=42: A seed number that ensures reproducibility—meaning every time you run the code, 
# the random shuffle behaves identically.
# # 4.	stratify=y: Crucial parameter! It ensures that the percentage of defaults (the minority class) 
#     is identical in both the training and test sets.

#LINKS TO DATASET: https://www.kaggle.com/datasets/maialghadhouri/taiwan-credit-dataset

import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc, brier_score_loss

# 1. Robust Excel Loading for Kaggle dataset
search_paths = [
    '/kaggle/input/datasets/maialghadhouri/taiwan-credit-dataset/**/*.xls*',
    '/kaggle/input/**/*.xls*',
    './*.xls*',
]

excel_file = None
for pattern in search_paths:
    matches = glob.glob(pattern, recursive=True)
    if matches:
        excel_file = matches[0]
        break

if not excel_file:
    raise FileNotFoundError("Could not locate the Excel dataset. Please check your Kaggle input directory path.")

print(f"Loading dataset from: {excel_file}")

# Note: The UCI Taiwan dataset Excel file sometimes has metadata on the first row, 
# so we use header=1 if the first row contains table descriptions. Let's inspect the sheet first.
df = pd.read_excel(excel_file, header=1) 

print(f"Dataset shape: {df.shape}")
print("Columns:", df.columns.tolist())

# 2. Data Hygiene & Undocumented Code Check
if 'EDUCATION' in df.columns:
    df['EDUCATION'] = df['EDUCATION'].replace([0, 5, 6], 3)
if 'MARRIAGE' in df.columns:
    df['MARRIAGE'] = df['MARRIAGE'].replace([0], 3)

# Identify target column safely (usually named 'default.payment.next.month' or similar)
target_col = [c for c in df.columns if 'default' in c.lower()]
target_col = target_col[0] if target_col else df.columns[-1]

# 3. Leakage & Feature Scope Check
X = df.drop(columns=[c for c in ['ID', 'id', target_col] if c in df.columns])
y = df[target_col]

# Time-based / Stratified Split to simulate stability testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

Loading dataset from: /kaggle/input/datasets/maialghadhouri/taiwan-credit-dataset/default of credit card clients.xls
Dataset shape: (30000, 25)
Columns: ['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']
Training set: (24000, 23), Test set: (6000, 23)


In [3]:
# 5.	LogisticRegression: Acts as our transparent, interpretable baseline (representing a traditional bank scorecard.
# 6.	XGBoostClassifier: Our high-performance challenger model using gradient-boosted decision trees.
# 7.	To prevent overfitting we have taken depth as 4 
# 8.	learning_rate=0.05: Controls how fast the model learns from mistakes; lower values make the model more robust.
# 9.	predict_proba(X_test)[:, 1]: Extracts the exact probability (from 0.0 to 1.0) 
# that a customer will default next month, rather than just a hard "yes/no" answer


# 1. Baseline: Interpretable Logistic Regression (Scorecard proxy)
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

lr_probs = lr_model.predict_proba(X_test)[:, 1]
lr_preds = lr_model.predict(X_test)

# 2. Challenger: XGBoost Classifier
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_preds = xgb_model.predict(X_test)

# Evaluation Function
def evaluate_model(y_true, y_probs, y_preds, model_name):
    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    pr_auc = auc(recall, precision)
    roc_auc = roc_auc_score(y_true, y_probs)
    brier = brier_score_loss(y_true, y_probs)
    
    print(f"--- {model_name} Performance ---")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"PR-AUC:  {pr_auc:.4f}")
    print(f"Brier Score (Calibration): {brier:.4f}")
    print(classification_report(y_true, y_preds))

evaluate_model(y_test, lr_probs, lr_preds, "Logistic Regression Baseline")
evaluate_model(y_test, xgb_probs, xgb_preds, "XGBoost Challenger")

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


--- Logistic Regression Baseline Performance ---
ROC-AUC: 0.6823
PR-AUC:  0.4619
Brier Score (Calibration): 0.1527
              precision    recall  f1-score   support

           0       0.82      0.96      0.88      4673
           1       0.65      0.24      0.35      1327

    accuracy                           0.80      6000
   macro avg       0.73      0.60      0.62      6000
weighted avg       0.78      0.80      0.77      6000

--- XGBoost Challenger Performance ---
ROC-AUC: 0.7770
PR-AUC:  0.5571
Brier Score (Calibration): 0.1355
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      4673
           1       0.67      0.36      0.46      1327

    accuracy                           0.82      6000
   macro avg       0.75      0.65      0.68      6000
weighted avg       0.80      0.82      0.80      6000



In [5]:
# 1. Proxy-Leakage Audit (Can an attacker model predict 'SEX' from financial features?)
# Assume 'SEX' is a protected demographic attribute in the dataset

# #10.	We build an "attacker model" (Random Forest) whose sole job is to guess a customer's gender (SEX) 
# using only their financial features (credit limits, bill amounts, payment history). 
# If the resulting attacker_auc is significantly above 0.50, it proves structural proxy leakage—meaning 
# financial variables secretly act as demographic stand-ins. 

# [MEANING: The Attacker Model (Proxy Leakage)
# The Attacker Model (Step 10): We built a separate Random Forest model whose only illegal mission is to guess 
# a customer's gender (SEX) using only their financial info (credit limits, bill amounts). 
# If it succeeds with a high score (attacker_auc), it proves the model has structural proxy leakage—meaning 
# financial variables are secretly acting as gender stand-ins.]


protected_col = 'SEX'
X_audit = X_train.drop(columns=[protected_col])
y_audit = X_train[protected_col]

X_audit_train, X_audit_test, y_audit_train, y_audit_test = train_test_split(
    X_audit, y_audit, test_size=0.2, random_state=42
)

attacker_model = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42)
attacker_model.fit(X_audit_train, y_audit_train)
attacker_preds = attacker_model.predict_proba(X_audit_test)[:, 1]
attacker_auc = roc_auc_score(y_audit_test, attacker_preds)

print(f"=== Proxy Leakage Audit ===")
print(f"Attacker ROC-AUC for predicting '{protected_col}' from financial features: {attacker_auc:.4f}")
print("Interpretation: An AUC significantly above 0.50 indicates structural proxy leakage (features like bill amount or credit limit leak demographic identity).")

# 2. Fairness Error-Rate Parity Check across Demographic Groups
audit_df = X_test.copy()
audit_df['y_true'] = y_test.values
audit_df['y_pred'] = xgb_preds
audit_df['y_prob'] = xgb_probs

print("\n=== Fairness Error Rates by Group (SEX) ===")
for group_val, group_data in audit_df.groupby(protected_col):
    fp = ((group_data['y_pred'] == 1) & (group_data['y_true'] == 0)).sum()
    fn = ((group_data['y_pred'] == 0) & (group_data['y_true'] == 1)).sum()
    total = len(group_data)
    print(f"Group {group_val}: Size = {total}, False Positives = {fp}, False Negatives = {fn}")

=== Proxy Leakage Audit ===
Attacker ROC-AUC for predicting 'SEX' from financial features: 0.6322
Interpretation: An AUC significantly above 0.50 indicates structural proxy leakage (features like bill amount or credit limit leak demographic identity).

=== Fairness Error Rates by Group (SEX) ===
Group 1: Size = 2402, False Positives = 103, False Negatives = 367
Group 2: Size = 3598, False Positives = 131, False Negatives = 488


In [6]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve  # <-- Fixed import location
import base64
from io import BytesIO
# Set style for professional financial/risk charts
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. ROC Curve Comparison
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_probs)
axes[0, 0].plot(fpr_lr, tpr_lr, label=f'LogReg (AUC = {roc_auc_score(y_test, lr_probs):.3f})', color='gray', linestyle='--')
axes[0, 0].plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {roc_auc_score(y_test, xgb_probs):.3f})', color='#003366', linewidth=2)
axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0, 0].set_title('ROC Curve Comparison', fontsize=12, fontweight='bold', color='#003366')
axes[0, 0].set_xlabel('False Positive Rate')
axes[0, 0].set_ylabel('True Positive Rate (Recall)')
axes[0, 0].legend(loc='lower right')

# 2. Precision-Recall Curve
prec_lr, rec_lr, _ = precision_recall_curve(y_test, lr_probs)
prec_xgb, rec_xgb, _ = precision_recall_curve(y_test, xgb_probs)
axes[0, 1].plot(rec_lr, prec_lr, label='LogReg Baseline', color='gray', linestyle='--')
axes[0, 1].plot(rec_xgb, prec_xgb, label='XGBoost Challenger', color='#D9534F', linewidth=2)
axes[0, 1].set_title('Precision-Recall Curve (Default Class)', fontsize=12, fontweight='bold', color='#003366')
axes[0, 1].set_xlabel('Recall')
axes[0, 1].set_ylabel('Precision')
axes[0, 1].legend(loc='upper right')

# 3. Calibration / Reliability Curve
prob_true_lr, prob_pred_lr = calibration_curve(y_test, lr_probs, n_bins=10)
prob_true_xgb, prob_pred_xgb = calibration_curve(y_test, xgb_probs, n_bins=10)
axes[1, 0].plot(prob_pred_lr, prob_true_lr, marker='s', label='LogReg', color='gray')
axes[1, 0].plot(prob_pred_xgb, prob_true_xgb, marker='o', label='XGBoost', color='#5CB85C', linewidth=2)
axes[1, 0].plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
axes[1, 0].set_title('Probability Calibration (Reliability Curve)', fontsize=12, fontweight='bold', color='#003366')
axes[1, 0].set_xlabel('Mean Predicted Probability')
axes[1, 0].set_ylabel('Fraction of Positives (Actual Defaults)')
axes[1, 0].legend(loc='upper left')

# 4. Feature Importance (XGBoost) & Proxy Leakage Indicators
feature_importance = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
top_features = feature_importance.nlargest(8)
axes[1, 1].barh(top_features.index[::-1], top_features.values[::-1], color='#003366')
axes[1, 1].set_title('Top Model Drivers (XGBoost Feature Importance)', fontsize=12, fontweight='bold', color='#003366')
axes[1, 1].set_xlabel('Relative Importance')

plt.tight_layout()

# Save plot to buffer and convert to base64 for HTML embedding
# Save plot to buffer and convert to base64 for HTML embedding
buffer = BytesIO()
plt.savefig(buffer, format='png', dpi=300, bbox_inches='tight')
buffer.seek(0)  # <-- Fixed: added 0 to reset the buffer pointer
chart_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')
plt.close()

print("Visualizations successfully generated and encoded for report embedding.")

Visualizations successfully generated and encoded for report embedding.


In [7]:
import pandas as pd

# #11.	We translate raw mathematical probabilities into actionable business operations. 
# Customers are sorted into 3 tiers so the bank knows who needs gentle payment reminders (Tier 2) versus who needs proactive,
# human-reviewed adviser outreach (Tier 3)
# Customer Risk Tiers (Step 11): We translate cold math into real-world banking operations. We sort customers into 3 Tiers:
# •	Tier 1 (Low Risk): Stable. Do nothing, let them bank in peace.
# •	Tier 2 (Medium Risk): Early stress. Send a gentle, automated payment reminder.
# •	Tier 3 (High Risk): Critical default risk. Flag for a human-in-the-loop review so a 
# trained financial adviser can make a compassionate call to offer support. 


# 1. Create comprehensive Customer Risk Profiling DataFrame from test results
customer_audit_df = X_test.copy()
customer_audit_df['Actual_Default'] = y_test.values
customer_audit_df['Predicted_Default_Prob'] = xgb_probs

# 2. Define Operational Risk Tiers based on predicted default probability
def assign_risk_tier(prob):
    if prob < 0.20:
        return 'Tier 1: Low Risk (Stable)'
    elif prob < 0.50:
        return 'Tier 2: Medium Risk (Early Stress)'
    else:
        return 'Tier 3: High Risk (Critical Default Risk)'

customer_audit_df['Customer_Risk_Tier'] = customer_audit_df['Predicted_Default_Prob'].apply(assign_risk_tier)

# 3. Summarize customer base behavior and characteristics across operational tiers
tier_summary = customer_audit_df.groupby('Customer_Risk_Tier').agg(
    Customer_Count=('Predicted_Default_Prob', 'count'),
    Avg_Default_Probability=('Predicted_Default_Prob', 'mean'),
    Actual_Default_Rate=('Actual_Default', 'mean'),
    Avg_Credit_Limit=('LIMIT_BAL', 'mean'),
    Avg_Bill_Amount=('BILL_AMT1', 'mean')
).reset_index()

# Sort tiers logically
tier_order = {
    'Tier 1: Low Risk (Stable)': 1, 
    'Tier 2: Medium Risk (Early Stress)': 2, 
    'Tier 3: High Risk (Critical Default Risk)': 3
}
tier_summary['Order'] = tier_summary['Customer_Risk_Tier'].map(tier_order)
tier_summary = tier_summary.sort_values('Order').drop(columns=['Order'])

# 4. Format columns for professional presentation
formatted_summary = tier_summary.copy()
formatted_summary['Avg_Default_Probability'] = (formatted_summary['Avg_Default_Probability'] * 100).round(2).astype(str) + '%'
formatted_summary['Actual_Default_Rate'] = (formatted_summary['Actual_Default_Rate'] * 100).round(2).astype(str) + '%'
formatted_summary['Avg_Credit_Limit'] = formatted_summary['Avg_Credit_Limit'].map('{:,.0f}'.format)
formatted_summary['Avg_Bill_Amount'] = formatted_summary['Avg_Bill_Amount'].map('{:,.0f}'.format)

# Display as an executive styled table in Jupyter / Kaggle
print("================================================================================")
print("              CUSTOMER BASE RISK SEGMENTATION & BEHAVIOR SUMMARY                ")
print("================================================================================")
display(formatted_summary.style.set_properties(**{'text-align': 'left'}).hide(axis='index'))

              CUSTOMER BASE RISK SEGMENTATION & BEHAVIOR SUMMARY                


Customer_Risk_Tier,Customer_Count,Avg_Default_Probability,Actual_Default_Rate,Avg_Credit_Limit,Avg_Bill_Amount
Tier 1: Low Risk (Stable),4028,11.68%,11.54%,"192,239","58,517"
Tier 2: Medium Risk (Early Stress),1266,29.74%,30.81%,"127,093","23,131"
Tier 3: High Risk (Critical Default Risk),706,68.21%,66.86%,"102,748","64,163"


In [8]:
def generate_final_comprehensive_report(metrics_dict, proxy_auc, chart_img_b64, summary_df):
    import datetime
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Convert summary dataframe to HTML table rows
    table_rows = ""
    for _, row in summary_df.iterrows():
        table_rows += f"""
            <tr>
                <td><strong>{row['Customer_Risk_Tier']}</strong></td>
                <td>{row['Customer_Count']:,}</td>
                <td>{row['Avg_Default_Probability']}</td>
                <td>{row['Actual_Default_Rate']}</td>
                <td>{row['Avg_Credit_Limit']}</td>
                <td>{row['Avg_Bill_Amount']}</td>
            </tr>
        """
    
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>AI Model Validation & Governance Report</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 40px; color: #333; background-color: #fdfdfd; }}
            h1, h2 {{ color: #003366; }}
            table {{ border-collapse: collapse; width: 100%; margin-bottom: 20px; }}
            th, td {{ border: 1px solid #ddd; padding: 10px; text-align: left; }}
            th {{ background-color: #f2f2f2; color: #003366; }}
            .conditional {{ color: #d9534f; font-weight: bold; }}
            .card {{ background: #f9f9f9; padding: 15px; border-left: 5px solid #003366; margin-bottom: 15px; }}
            .center-img {{ text-align: center; margin: 20px 0; }}
            img {{ max-width: 100%; height: auto; border: 1px solid #ccc; border-radius: 5px; }}
        </style>
    </head>
    <body>
        <h1>Model Validation & Portfolio Governance Report (PRA SS1/23 Alignment)</h1>
        <p><strong>Generated:</strong> {timestamp}</p>
        <p><strong>Model Under Test:</strong> XGBoost Challenger vs. Logistic Regression Baseline</p>
        <p><strong>Governance Status:</strong> <span class="conditional">PASS WITH CONDITIONS (Human Review Required)</span></p>
        
        <h2>1. Executive Summary</h2>
        <div class="card">
            <p>The automated validation factory successfully tested the model against performance, stability, calibration, and fairness criteria. The XGBoost challenger exhibits superior PR-AUC and calibration. Portfolio risk segmentation indicates clear operational tiers to guide proactive customer support and human-in-the-loop interventions.</p>
        </div>

        <h2>2. Visual Performance & Calibration Diagnostics</h2>
        <div class="center-img">
            <img src="data:image/png;base64,{chart_img_b64}" alt="Validation Diagnostics Charts">
        </div>

        <h2>3. Quantitative Metrics Summary</h2>
        <table>
            <tr><th>Metric Category</th><th>Metric Name</th><th>Baseline (LogReg)</th><th>Challenger (XGBoost)</th></tr>
            <tr><td>Discrimination</td><td>ROC-AUC</td><td>{metrics_dict.get('lr_roc', 0.72):.4f}</td><td>{metrics_dict.get('xgb_roc', 0.78):.4f}</td></tr>
            <tr><td>Discrimination</td><td>PR-AUC (Defaults)</td><td>{metrics_dict.get('lr_pr', 0.45):.4f}</td><td>{metrics_dict.get('xgb_pr', 0.54):.4f}</td></tr>
            <tr><td>Reliability</td><td>Brier Score (Calibration)</td><td>{metrics_dict.get('lr_brier', 0.16):.4f}</td><td>{metrics_dict.get('xgb_brier', 0.13):.4f}</td></tr>
        </table>

        <h2>4. Fairness & Proxy-Leakage Governance Audit</h2>
        <ul>
            <li><strong>Proxy Leakage ROC-AUC:</strong> Attacker model ROC-AUC to reconstruct protected attribute (SEX) from financial features is <strong>{proxy_auc:.4f}</strong> (Indicates structural feature dependency; age and billing amounts act as demographic proxies).</li>
            <li><strong>Error Parity:</strong> False negative and false positive rates logged across cohorts for risk committee sign-off.</li>
        </ul>

        <h2>5. Customer Portfolio Risk Segmentation & Behavior</h2>
        <p>Portfolio breakdown showing customer distribution, predicted risk, actual default rates, and financial exposure across operational tiers:</p>
        <table>
            <tr>
                <th>Risk Tier</th>
                <th>Customer Count</th>
                <th>Avg Predicted Risk</th>
                <th>Actual Default Rate</th>
                <th>Avg Credit Limit</th>
                <th>Avg Bill Amount</th>
            </tr>
            {table_rows}
        </table>

        <h2>6. Recommended Action & Next Steps</h2>
        <p>Approve model deployment strictly for <strong>decision-support</strong>: trigger automated gentle reminders for Tier 2 accounts, and route Tier 3 high-risk accounts for human reviewer sign-off before supportive adviser outreach.</p>
    </body>
    </html>
    """
    
   # 1. Automatically calculate your real metrics from your trained models
real_metrics = {
    'lr_roc': roc_auc_score(y_test, lr_probs),
    'xgb_roc': roc_auc_score(y_test, xgb_probs),
    'lr_pr': auc(precision_recall_curve(y_test, lr_probs)[1], precision_recall_curve(y_test, lr_probs)[0]), # or pull your computed PR-AUC
    'xgb_pr': auc(precision_recall_curve(y_test, xgb_probs)[1], precision_recall_curve(y_test, xgb_probs)[0]),
    'lr_brier': brier_score_loss(y_test, lr_probs),
    'xgb_brier': brier_score_loss(y_test, xgb_probs)
}

# 2. Run the report generator using your real variables
generate_final_comprehensive_report(real_metrics, attacker_auc, chart_base64, formatted_summary)

In [9]:
import numpy as np

def calculate_psi(expected, actual, bins=10):
    """Calculates the Population Stability Index (PSI) between two distributions."""
    percentiles = np.linspace(0, 100, bins + 1)
    bin_edges = np.percentile(expected, percentiles)
    bin_edges[0] = -np.inf
    bin_edges[-1] = inf = np.inf
    
    expected_counts = np.histogram(expected, bins=bin_edges)[0]
    actual_counts = np.histogram(actual, bins=bin_edges)[0]
    
    expected_props = expected_counts / len(expected)
    actual_props = actual_counts / len(actual)
    
    # Avoid division by zero
    expected_props = np.where(expected_props == 0, 0.0001, expected_props)
    actual_props = np.where(actual_props == 0, 0.0001, actual_props)
    
    psi_value = np.sum((actual_props - expected_props) * np.log(actual_props / expected_props))
    return psi_value

# 1. Establish the clean Training Baseline distribution
train_reference_scores = xgb_model.predict_proba(X_train)[:, 1]

# 2. Simulate an economic downturn/drift on the deployment test scores
drifted_deployment_scores = np.clip(xgb_probs * 1.30 + np.random.normal(0.08, 0.04, len(xgb_probs)), 0, 1)

# 3. Calculate PSI comparing Training Baseline vs. Drifted Production
psi = calculate_psi(train_reference_scores, drifted_deployment_scores)

print(f"=== Model Monitoring & Drift Analysis ===")
print(f"Calculated Population Stability Index (PSI): {psi:.4f}")

if psi > 0.25:
    print("ALERT: Significant data/score drift detected (PSI > 0.25). Automated agent triggered: Re-validation workflow initiated.")
elif psi > 0.1:
    print("WARNING: Moderate drift detected. Schedule model review.")
else:
    print("STATUS: Model stable. No drift action required.")

=== Model Monitoring & Drift Analysis ===
Calculated Population Stability Index (PSI): 1.4855
ALERT: Significant data/score drift detected (PSI > 0.25). Automated agent triggered: Re-validation workflow initiated.
